# Validación y métrica del negocio de mi proyecto

Evalúo mi modelo final (bosque aleatorio) y el segundo mejor de los que aprenden (red neuronal) con validación cruzada. La "línea base (siempre 0)" va solo como referencia.

Solo uso los datos anteriores a 2016: el año 2016 sigue siendo mi conjunto de prueba y la validación cruzada no lo toca.

**Limitación:** `precio_relativo` se calculó con el precio promedio de cada producto hasta 2015. En un pliegue temporal, ese promedio incluye meses posteriores al pliegue, así que hay una pequeña fuga de información que no pude quitar sin recalcular la feature sobre los 59 millones de filas originales.

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from pipeline_proyecto import X_ent, datos, es_entrenamiento, preparacion, y_ent

# Los mismos datos de entrenamiento, del dia mas antiguo al mas reciente.
fechas = datos.loc[es_entrenamiento, "DATE"]
orden = fechas.sort_values(kind="stable").index
X_orden, y_orden, fechas_orden = X_ent.loc[orden], y_ent.loc[orden], fechas.loc[orden]
print("Filas:", len(X_orden), "| desde", fechas_orden.iloc[0], "hasta", fechas_orden.iloc[-1])

Filas y columnas del archivo: (468820, 15)
Entrenamiento: fechas 2011-01-29 a 2015-12-31
Prueba: fechas 2016-01-01 a 2016-05-22
Vacios en entrenamiento: {}
Vacios en prueba: {'precio_relativo': 21}


Entrenamiento: (424975, 26) Prueba: (43845, 26)
['numericas__SELL_PRICE', 'numericas__precio_relativo', 'numericas__dias_desde_lanzamiento', 'categoricas__CAT_ID_FOODS', 'categoricas__CAT_ID_HOBBIES', 'categoricas__CAT_ID_HOUSEHOLD', 'categoricas__STORE_ID_CA_1', 'categoricas__STORE_ID_CA_2', 'categoricas__STORE_ID_CA_3', 'categoricas__STORE_ID_CA_4', 'categoricas__STORE_ID_TX_1', 'categoricas__STORE_ID_TX_2', 'categoricas__STORE_ID_TX_3', 'categoricas__STORE_ID_WI_1', 'categoricas__STORE_ID_WI_2', 'categoricas__STORE_ID_WI_3', 'categoricas__WEEKDAY_Friday', 'categoricas__WEEKDAY_Monday', 'categoricas__WEEKDAY_Saturday', 'categoricas__WEEKDAY_Sunday', 'categoricas__WEEKDAY_Thursday', 'categoricas__WEEKDAY_Tuesday', 'categoricas__WEEKDAY_Wednesday', 'ya_listas__IS_WEEKEND', 'ya_listas__IS_EVENT_DAY', 'ya_listas__IS_SNAP']
Medianas que aprendio el imputador: {'SELL_PRICE': 3.44, 'precio_relativo': 1.0, 'dias_desde_lanzamiento': 729.0}
Medianas de entrenamiento: {'SELL_PRICE': 3.44, 'prec

Filas: 424975 | desde 2011-01-29 hasta 2015-12-31


## La métrica del negocio

Predigo las unidades de cada producto en cada tienda para saber cuánto stock tener. **Subestimar es más caro:** si faltan unidades pierdo la venta completa y su margen, y el cliente puede irse a otra tienda. Sobreestimar deja unidades que normalmente se venden después, y solo cuesta almacenarlas (algo más en alimentos, que caducan).

Supongo, de forma aproximada, que **una unidad que falta cuesta 3 veces lo que cuesta una unidad que sobra**. El 3 es una suposición mía, no un dato de la empresa.

In [2]:
COSTO_FALTANTE = 3
COSTO_SOBRANTE = 1


def costo_stock(y_real, y_estimada):
    """Costo promedio por fila, en unidades equivalentes de stock sobrante."""
    diferencia = np.asarray(y_real) - np.asarray(y_estimada)
    return np.mean(COSTO_FALTANTE * np.maximum(diferencia, 0) + COSTO_SOBRANTE * np.maximum(-diferencia, 0))


METRICAS = {"mae": "neg_mean_absolute_error", "costo": make_scorer(costo_stock, greater_is_better=False)}

## Los dos esquemas

- **Temporal (el que corresponde):** predigo el futuro, así que cada pliegue entrena con días anteriores y valida con los siguientes. Parto por días, para que un mismo día nunca quede a los dos lados.
- **Al azar (para comparar):** 5 pliegues mezclados.

In [3]:
dias = np.array(sorted(fechas_orden.unique()))
fechas_valores = fechas_orden.to_numpy()
pliegues_temporales = []
for dias_ent, dias_val in TimeSeriesSplit(n_splits=5).split(dias):
    ent = np.flatnonzero(np.isin(fechas_valores, dias[dias_ent]))
    val = np.flatnonzero(np.isin(fechas_valores, dias[dias_val]))
    pliegues_temporales.append((ent, val))
    print(f"entrena {dias[dias_ent][0]} a {dias[dias_ent][-1]} ({len(ent):6d} filas)   valida {dias[dias_val][0]} a {dias[dias_val][-1]} ({len(val):6d} filas)")

ESQUEMAS = {"temporal": pliegues_temporales, "al azar": KFold(n_splits=5, shuffle=True, random_state=42)}

entrena 2011-01-29 a 2011-11-27 ( 45293 filas)   valida 2011-11-28 a 2012-09-21 ( 56253 filas)


entrena 2011-01-29 a 2012-09-21 (101546 filas)   valida 2012-09-22 a 2013-07-17 ( 67537 filas)


entrena 2011-01-29 a 2013-07-17 (169083 filas)   valida 2013-07-18 a 2014-05-12 ( 78368 filas)


entrena 2011-01-29 a 2014-05-12 (247451 filas)   valida 2014-05-13 a 2015-03-07 ( 86829 filas)


entrena 2011-01-29 a 2015-03-07 (334280 filas)   valida 2015-03-08 a 2015-12-31 ( 90695 filas)


## Validación cruzada

In [4]:
MODELOS = {
    "linea base (siempre 0)": DummyRegressor(strategy="median"),
    "bosque aleatorio (final)": RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=42),
    "red neuronal (segundo)": MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, random_state=42),
}

filas = []
for nombre_esquema, esquema in ESQUEMAS.items():
    for nombre_modelo, modelo in MODELOS.items():
        pipeline = Pipeline([("preparar", clone(preparacion)), ("modelo", modelo)])
        puntajes = cross_validate(pipeline, X_orden, y_orden, cv=esquema, scoring=METRICAS)
        for metrica in METRICAS:
            valores = -puntajes["test_" + metrica]
            fila = {"esquema": nombre_esquema, "modelo": nombre_modelo, "metrica": metrica,
                    "promedio": valores.mean(), "desviacion": valores.std()}
            fila.update({f"pliegue_{i + 1}": v for i, v in enumerate(valores)})
            filas.append(fila)
        print("Listo:", nombre_esquema, "|", nombre_modelo, "|", round(puntajes["fit_time"].sum()), "s", flush=True)

tabla = pd.DataFrame(filas)
tabla.to_csv("validacion_proyecto.csv", index=False)
print(tabla.round(4).to_string(index=False))

Listo: temporal | linea base (siempre 0) | 1 s


Listo: temporal | bosque aleatorio (final) | 148 s


/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (7 outputs left out)

/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Listo: al azar | red neuronal (segundo) | 1017 s


 esquema                   modelo metrica  promedio  desviacion  pliegue_1  pliegue_2  pliegue_3  pliegue_4  pliegue_5
temporal   linea base (siempre 0)     mae    1.4321      0.1857     1.7246     1.5449     1.4102     1.2313     1.2495
temporal   linea base (siempre 0)   costo    4.2963      0.5571     5.1738     4.6346     4.2305     3.6940     3.7486
temporal bosque aleatorio (final)     mae    1.6855      0.1844     1.9456     1.8104     1.7082     1.5215     1.4418
temporal bosque aleatorio (final)   costo    3.2217      0.4077     3.8983     3.4285     3.1438     2.8184     2.8196
temporal   red neuronal (segundo)     mae    1.8832      0.2290     2.1641     2.0990     1.8917     1.6989     1.5625
temporal   red neuronal (segundo)   costo    3.4773      0.4127     4.0635     3.8280     3.4132     3.0346     3.0473
 al azar   linea base (siempre 0)     mae    1.4370      0.0068     1.4239     1.4393     1.4393     1.4438     1.4390
 al azar   linea base (siempre 0)   costo    4.3

/home/vscode/.local/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


## El modelo final contra el segundo, pliegue por pliegue

In [5]:
PLIEGUES = [f"pliegue_{i}" for i in range(1, 6)]

for esquema in ESQUEMAS:
    for metrica in METRICAS:
        filtro = (tabla["esquema"] == esquema) & (tabla["metrica"] == metrica)
        final = tabla[filtro & (tabla["modelo"] == "bosque aleatorio (final)")][PLIEGUES].to_numpy()[0]
        segundo = tabla[filtro & (tabla["modelo"] == "red neuronal (segundo)")][PLIEGUES].to_numpy()[0]
        diferencia = segundo - final
        print(f"{esquema:9s} {metrica:6s} ventaja del bosque: promedio {diferencia.mean():7.4f}  desviacion {diferencia.std():.4f}  "
              f"gana en {(diferencia > 0).sum()} de 5 pliegues  (desviacion entre pliegues del bosque: {final.std():.4f})")

temporal  mae    ventaja del bosque: promedio  0.1977  desviacion 0.0552  gana en 5 de 5 pliegues  (desviacion entre pliegues del bosque: 0.1844)
temporal  costo  ventaja del bosque: promedio  0.2556  desviacion 0.0793  gana en 5 de 5 pliegues  (desviacion entre pliegues del bosque: 0.4077)
al azar   mae    ventaja del bosque: promedio  0.1142  desviacion 0.0241  gana en 5 de 5 pliegues  (desviacion entre pliegues del bosque: 0.0062)
al azar   costo  ventaja del bosque: promedio  0.2392  desviacion 0.0071  gana en 5 de 5 pliegues  (desviacion entre pliegues del bosque: 0.0179)
